In [26]:
import pandas as pd
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder


In [27]:
df=pd.read_csv('loan_data.csv.xls')

In [28]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [29]:
x=df.drop(columns='loan_status')
y=df.loan_status

In [30]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42)

In [31]:
num_col=x.select_dtypes(include='number').columns
obj_col=x.select_dtypes(include='object').columns

C:\Users\Admin\AppData\Local\Temp\ipykernel_1092\2045516051.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_col=x.select_dtypes(include='object').columns


In [32]:
x[obj_col].nunique()

person_gender                     2
person_education                  5
person_home_ownership             4
loan_intent                       6
previous_loan_defaults_on_file    2
dtype: int64

In [35]:
x['person_education'].unique()

<ArrowStringArray>
['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']
Length: 5, dtype: str

In [37]:
obj_col.drop('person_education')

Index(['person_gender', 'person_home_ownership', 'loan_intent',
       'previous_loan_defaults_on_file'],
      dtype='str')

In [36]:
order=['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate']

In [ ]:
preprocessing = ColumnTransformer(
    transformers=[
        ('onehot_encoder',OneHotEncoder(handle_unknown='ignore'),obj_col.drop('person_education')),
        ('ordinal',OrdinalEncoder(categories=[order],handle_unknown='use_encoded_value',unknown_value=-1),['person_education'])
         ],remainder='passthrough'

)
main_pipeline=Pipeline(
    steps=[
        ('preprocessing',preprocessing),
        ('model',DecisionTreeClassifier(random_state=42))
    ]
) 
#whenever we use pipeline use model__ parameter name
grid_search_cv=GridSearchCV(
    estimator=main_pipeline,
    param_grid={
        'model__criterion':['gini','entropy'],
        'model__max_depth':[None,5,10,50,100],
        'model__min_samples_split':[2,5,7,10],
        'model__min_samples_leaf':[1,3,5,7,10],
        'model__splitter':['best','random']
    }
)    
grid_search_cv.fit(xtrain,ytrain)     
                
 